# Figure 2 and Supplementary Figures 2-6: LN module score

The MICE Cox models behind SFig2 are fitted in
`code/01_compute/04_clinical_adjustment_mice.ipynb`; this notebook only plots their results.

**Inputs**
- input_dir `metadata/sample_metadata.tsv` (91 cfRNA samples) and `metadata/multisource_metadata.tsv` (91 cfRNA + 20 TME + 17 PBMC)
- input_dir `counts/counts_TMMCPM_cart_paper_ALLDATA.tsv.gz` (gene symbols; Fig2, SFig3A, SFig6A)
- input_dir `counts/counts_norm_cart_paper.tsv.gz` + `reference/gencode.biotype.name.key.tsv.gz` (SFig4)
- input_dir `scrna/maurer2024_pbmc_baseline_seurat.rds` + `scrna/maurer2024_pbmc_response_key.tsv` (SFig5; Maurer et al. 2024)
- results `mice/sfig2_univariate_cox.csv` and `mice/sfig2_scores_multivariable_cox.csv` (SFig2)

**Outputs** (plots/)
- `Fig2AB_ln_module.pdf`: A) LN score by response across TME / cfRNA / PBMC; B) PFS by high/low cfRNA LN score within each cohort
- `SFig2_univariate_cox_forest.pdf`: MICE-pooled Cox hazard ratios of the gene-set scores and the clinical variables
- `SFig3A_ln_by_lbcl_subtype.pdf`: LN score by response in de novo DLBCL and other LBCL
- `SFig4_ln_vs_ldh.pdf`: LN score vs LDH (pre-lymphodepletion and Day -30)
- `SFig5_scrna_archetype_scores.pdf`: archetype module scores across PBMC cell types (Maurer et al. 2024)
- `SFig6A_fmac_tex_scores.pdf`: FMAC and TEX scores by response across sources
- `SFig6B_pfs_by_lymphomap.pdf`: PFS by LymphoMAP archetype

Gene sets come from `utils/genesets.yaml`.

In [ ]:
source("../../utils/setup.R")
library(survminer)
library(forestplot)
suppressPackageStartupMessages(library(Seurat))

## Load data

In [ ]:
mdf <- load_sample_metadata()              # 91 cfRNA samples
ms_meta <- load_multisource_metadata()     # 128 cfRNA / TME / PBMC samples

counts_tmmcpm <- load_counts("tmmcpm_all") # symbols x 130 samples (91 cfRNA + 17 PBMC + 22 TME)
counts_norm <- load_counts("norm")         # versioned Ensembl IDs x 91 cfRNA samples
gene_key <- load_gene_key()

# Maurer et al. 2024 baseline PBMC scRNA-seq (SFig5)
maurer_rds <- file.path(INPUT_DIR, "scrna", "maurer2024_pbmc_baseline_seurat.rds")
maurer_key <- file.path(INPUT_DIR, "scrna", "maurer2024_pbmc_response_key.tsv")
HAS_MAURER <- file.exists(maurer_rds) && file.exists(maurer_key)
if (HAS_MAURER) {
    seu <- readRDS(maurer_rds)
    response_key <- read.delim(maurer_key, stringsAsFactors = FALSE)
} else {
    message("PBMC object not found in input_dir/scrna/: skipping Supplementary Figure 5.")
}

cox_univariate <- read.csv(res_in("mice", "sfig2_univariate_cox.csv"))
cox_scores <- read.csv(res_in("mice", "sfig2_scores_multivariable_cox.csv"))

## Archetype scores across sources

`AddGeneSetScore` (nbin = 25, ctrl = 100, seed = 123) on the TMM-CPM matrix of all 128 samples. The scores are min-max
scaled across these 128 samples; they are used in Fig2A, Fig2B and SFig6A.

In [ ]:
x_all <- counts_tmmcpm[, ms_meta$sample_id]

all_scores <- cbind(AddGeneSetScore(x_all, LN_SYMBOLS, name = "LN_set"),
                    AddGeneSetScore(x_all, FMAC_SYMBOLS, name = "FMAC_set"),
                    AddGeneSetScore(x_all, TEX_SYMBOLS, name = "TEX_set")) %>%
    rownames_to_column("sample_id") %>%
    left_join(ms_meta, by = "sample_id") %>%
    # panel order of the figures: TME, cfRNA, PBMC
    mutate(dataset = factor(dataset, levels = c("TME", "cfRNA", "PBMC")))

table(all_scores$dataset)

## Figure 2A: LN score by response across sources

Box plots by response (R vs NR) within each cohort; TME biopsies (Li et al. 2025), plasma cfRNA (cohorts 1-3) and PBMCs (Maurer et al. 2024).
P-values: two-sided Wilcoxon rank-sum test, R vs NR within each cohort (normal approximation, `exact = FALSE`, as in all
Wilcoxon tests of the paper), labelled in ggpubr's format.

In [ ]:
ln_pvalues <- all_scores %>%
    group_by(dataset, cohort) %>%
    summarise(n_R = sum(response == "R"), n_NR = sum(response == "NR"),
              p = wilcox.test(LN_set1[response == "R"], LN_set1[response == "NR"], exact = FALSE)$p.value,
              .groups = "drop")
ln_pvalues

# One bracket per cohort, from the NR to the R box (dodge 0.8 -> centres at x -/+ 0.2), just above the
# highest point of the panel. Labels as ggpubr prints them: format.pval(digits = 2) over one panel's p-values.
ln_brackets <- all_scores %>%
    group_by(dataset) %>%
    mutate(x = as.numeric(factor(cohort)), y.position = max(LN_set1) + 0.05 * diff(range(LN_set1))) %>%
    group_by(dataset, cohort) %>%
    summarise(xmin = first(x) - 0.2, xmax = first(x) + 0.2, y.position = first(y.position), .groups = "drop") %>%
    left_join(ln_pvalues, by = c("dataset", "cohort")) %>%
    group_by(dataset) %>%
    mutate(label = format.pval(p, digits = 2)) %>%
    ungroup()

ln_box <- response_boxplot(all_scores, x = "cohort", y = "LN_set1", y_lab = "LN Score",
                           facet = "dataset", strip_size = 8) +
    geom_bracket(data = ln_brackets, aes(xmin = xmin, xmax = xmax, y.position = y.position, label = label),
                 inherit.aes = FALSE, tip.length = 0, size = 0.5, label.size = 2.5, vjust = -0.2) +
    scale_y_continuous(expand = expansion(mult = c(0.05, 0.15)))
ln_box

## Figure 2B: PFS by high/low cfRNA LN score within each cohort

Split at the median cfRNA LN score of all 91 cfRNA samples. Solid = low, dashed = high; colour = cohort. Patients infused
before 2024-12-01 (`km_include`).

In [ ]:
ln_cfrna <- all_scores %>% filter(dataset == "cfRNA") %>% select(sample_id, LN_set1)
ln_threshold <- median(ln_cfrna$LN_set1)

km_ln_df <- mdf %>%
    inner_join(ln_cfrna, by = "sample_id") %>%
    filter(km_include) %>%
    mutate(ln_group = ifelse(LN_set1 < ln_threshold, "Low", "High"),
           group = factor(paste0(cohort, " - ", ln_group),
                          levels = paste(rep(COHORT_LEVELS, each = 2), c("Low", "High"), sep = " - ")))

km_ln <- km_by_group(km_ln_df, group = "group",
                     palette = rep(COHORT_COLORS, each = 2),
                     linetype = rep(c("solid", "twodash"), 3),
                     major_grid = FALSE)
km_ln

In [ ]:
fig2 <- (ln_box + theme_axis_lines()) + plot_spacer() +
    (km_ln$plot + theme_axis_lines()) +
    plot_layout(nrow = 1, widths = c(3, 0.1, 1))
fig2
save_plot(fig2, "Fig2AB_ln_module", width = 6.95, height = 2)

## Supplementary Figure 2: univariate Cox regression (MICE)

MICE-pooled Cox hazard ratios for PFS (85 cfRNA patients infused before 2024-12-01, 20 imputations): one model with the
three per-SD gene-set scores together, and one univariate model per clinical variable. Fitted in
`01_compute/04_clinical_adjustment_mice.ipynb`.

In [ ]:
label_map <- c(
    LN_set1_z = "LN Score", FMAC_set1_z = "FMAC Score", TEX_set1_z = "TEX Score",
    lines_of_therapy = "Lines of Therapy", IPI_prelymphodepletion = "IPI Pre-Lymphodepletion",
    LDH_prelymphodepletion = "LDH Pre-Lymphodepletion", stage_advanced = "Stage Advanced (III-IV)",
    stage_early = "Stage Early (I-II) vs III-IV",
    LDH_Day30 = "LDH Day 30", De_novo_DLBCL = "De novo DLBCL", Timepoint = "Timepoint",
    Bridging = "Bridging"
)

section <- function(label, df) {
    bind_rows(tibble(term = label, is_header = TRUE),
              df %>% mutate(is_header = FALSE))
}

forest_rows <- bind_rows(
    section("Gene-set Scores",
            cox_scores %>% transmute(term = recode(term, !!!label_map), estimate, conf.low, conf.high, p.value)),
    section("Clinical Variables",
            cox_univariate %>%
                # stage_advanced has reference level "Advanced (III-IV)", so its HR is Early vs Advanced
                mutate(variable = ifelse(term == "stage_advancedEarly (I-II)", "stage_early", variable)) %>%
                transmute(term = recode(variable, !!!label_map), estimate, conf.low, conf.high, p.value) %>%
                mutate(across(c(estimate, conf.low, conf.high), ~ ifelse(is.finite(.x) & .x > 0, .x, NA_real_))))
)
forest_rows

In [ ]:
hr_col <- ifelse(is.na(forest_rows$estimate), "",
                 sprintf("%.2f (%.2f-%.2f)", forest_rows$estimate, forest_rows$conf.low, forest_rows$conf.high))
p_col <- ifelse(is.na(forest_rows$p.value), "",
                ifelse(forest_rows$p.value < 0.001, "<0.001", sprintf("%.3f", forest_rows$p.value)))
labeltext <- rbind(c("Variable", "HR (95% CI)", "p-value"),
                   cbind(forest_rows$term, hr_col, p_col))

cox_forest <- forestplot(
    labeltext  = labeltext,
    mean       = c(NA, forest_rows$estimate),
    lower      = c(NA, forest_rows$conf.low),
    upper      = c(NA, forest_rows$conf.high),
    is.summary = c(TRUE, forest_rows$is_header),
    graph.pos  = 3,
    xlog       = TRUE,
    zero       = 1,
    xticks     = c(0.1, 0.3, 1, 3, 10),
    clip       = c(0.1, 10),  # CIs beyond the axis end in an arrow
    boxsize    = 0.18,
    lwd.ci     = 1.6,
    lwd.zero   = 1,
    col        = fpColors(box = FOREST_BLUE, line = FOREST_BLUE, zero = "grey40", summary = "black"),
    hrzl_lines = list("2" = gpar(lty = 1, lwd = 1, col = "black")),  # line under the column headers
    xlab       = "Hazard Ratio",
    txt_gp     = fpTxtGp(
        label   = gpar(fontfamily = "Helvetica", cex = 0.8),
        ticks   = gpar(fontfamily = "Helvetica", cex = 0.75),
        xlab    = gpar(fontfamily = "Helvetica", cex = 0.85, fontface = "bold"),
        title   = gpar(fontfamily = "Helvetica", cex = 1, fontface = "bold"),
        summary = gpar(fontfamily = "Helvetica", cex = 0.85, fontface = "bold")
    ),
    align      = c("l", "c", "c", "c")
)
cox_forest

save_plot(grid::grid.grabExpr(print(cox_forest)), "SFig2_univariate_cox_forest",
          width = 6.5, height = 0.3 * nrow(labeltext) + 1.2)

## Supplementary Figure 3A: LN score by LBCL subtype

The LN score is recomputed on the 91 cfRNA samples only (TMM-CPM, nbin = 25), so it is min-max scaled within cfRNA.
P-values: two-sided Wilcoxon rank-sum test (normal approximation), R vs NR within each subtype.

In [ ]:
# Clinical-file row order (sets the jitter positions)
mdf_clin <- mdf[order(mdf$row_order_clinical), ]

ln_subtype <- AddGeneSetScore(counts_tmmcpm[, mdf_clin$sample_id], LN_SYMBOLS, name = "LN_set") %>%
    rownames_to_column("sample_id") %>%
    left_join(mdf_clin, by = "sample_id") %>%
    mutate(Lymphoma = ifelse(lymphoma_malignancy == "De novo DLBCL", "De novo DLBCL", "Non De novo DLBCL"))
table(ln_subtype$Lymphoma, ln_subtype$response)

sfig3a <- response_boxplot(ln_subtype, x = "Lymphoma", y = "LN_set1", y_lab = "LN Score",
                           facet = "Lymphoma", strip_size = 6, strip_margin = TRUE,
                           show_title = FALSE, stat_method = "wilcox.test")
sfig3a
save_plot(sfig3a, "SFig3A_ln_by_lbcl_subtype", width = 2.5, height = 2)

# The p-values printed on the panel
ggpubr::compare_means(LN_set1 ~ response, data = ln_subtype, group.by = "Lymphoma", method = "wilcox.test",
                      exact = FALSE)

## Supplementary Figure 4: LN score vs LDH

LN score on the normalized counts (`counts_norm`, Ensembl IDs mapped to symbols; nbin = 6 because 25 bins cannot be
formed on this matrix), min-max scaled to 0-1 over the 91 samples as returned by `AddGeneSetScore` (not z-scored; the
correlation is the same either way). LDH values are log2-transformed. Pearson correlation.

In [ ]:
counts_norm_sym <- ensembl_to_symbol(counts_norm, gene_key)
ln_norm <- AddGeneSetScore(counts_norm_sym[, mdf$sample_id], LN_SYMBOLS, nbin = 6, ctrl = 100, name = "LN_set") %>%
    rownames_to_column("sample_id")

ldh_df <- recode_clinical(mdf, "baseline") %>%
    inner_join(ln_norm, by = "sample_id")

plot_ln_vs_ldh <- function(df, ldh_var, x_lab) {
    sub <- df %>% select(LN_set1, all_of(ldh_var)) %>% drop_na()
    ct <- cor.test(sub$LN_set1, sub[[ldh_var]], method = "pearson")
    p <- ggplot(sub, aes(x = .data[[ldh_var]], y = LN_set1)) +
        geom_point(alpha = 0.75, color = RESPONSE_COLORS[["R"]], stroke = NA, size = 1.5) +
        geom_smooth(method = "lm", formula = y ~ x, se = TRUE, color = GENESET_COLOR, linewidth = 0.6) +
        annotate("text", x = Inf, y = Inf, hjust = 1.1, vjust = 3, size = 2.5,
                 label = sprintf("r= %.2f, p=%.3g", ct$estimate, ct$p.value)) +
        labs(x = x_lab, y = "cfRNA LN Score") +
        theme_cfrna_print() +
        theme_axis_lines()
    list(cor_test = ct, plot = p)
}

ln_vs_ldh_pre <- plot_ln_vs_ldh(ldh_df, "LDH", "LDH pre-lymphodepletion (log2)")
ln_vs_ldh_d30 <- plot_ln_vs_ldh(ldh_df, "LDH_D30", "LDH Day-30 (log2)")
print(ln_vs_ldh_pre$cor_test)
print(ln_vs_ldh_d30$cor_test)

sfig4 <- ln_vs_ldh_pre$plot + ln_vs_ldh_d30$plot
sfig4
save_plot(sfig4, "SFig4_ln_vs_ldh", width = 4.5, height = 2)

## Supplementary Figure 5: archetype scores across PBMC cell types

Maurer et al. 2024 baseline PBMC scRNA-seq. `Seurat::AddModuleScore` (default seed) scores each cell; scores are
averaged per sample and cell type (`Labels`), and sample-cell type pairs with fewer than 10 cells are removed.

In [ ]:
if (HAS_MAURER) {
    seu$response <- response_key$response[match(seu$Sample, response_key$Sample)]

    cat("FMAC genes found:", sum(FMAC_SYMBOLS %in% rownames(seu)), "/", length(FMAC_SYMBOLS), "\n")
    cat("LN genes found:",   sum(LN_SYMBOLS %in% rownames(seu)),   "/", length(LN_SYMBOLS), "\n")
    cat("TEX genes found:",  sum(TEX_SYMBOLS %in% rownames(seu)),  "/", length(TEX_SYMBOLS), "\n")

    # AddModuleScore appends the set index to each name (FMAC_score1, LN_score2, TEX_score3)
    seu <- AddModuleScore(seu, features = list(FMAC = FMAC_SYMBOLS, LN = LN_SYMBOLS, TEX = TEX_SYMBOLS),
                          name = c("FMAC_score", "LN_score", "TEX_score"))

    pb <- seu@meta.data %>%
        filter(!is.na(response)) %>%
        group_by(sample_id = Sample, celltype = Labels, response) %>%
        summarise(n_cells = n(),
                  FMAC_score = mean(FMAC_score1), LN_score = mean(LN_score2), TEX_score = mean(TEX_score3),
                  .groups = "drop") %>%
        filter(n_cells >= 10)  # response kept as character (sets the jitter positions)
    print(head(pb))
}

In [ ]:
if (HAS_MAURER) {
    pbmc_box <- function(score) {
        response_boxplot(pb, x = "celltype", y = score, y_lab = score, jitter_width = 0.2,
                         title_size = 8, axis_lines = TRUE)
    }

    sfig5 <- (pbmc_box("FMAC_score") / pbmc_box("LN_score") / pbmc_box("TEX_score")) +
        plot_layout(guides = "collect") & theme(legend.position = "bottom")
    print(sfig5)
    save_plot(sfig5, "SFig5_scrna_archetype_scores", width = 6, height = 5)
}

## Supplementary Figure 6A: FMAC and TEX scores across sources

Same scores and layout as Fig2A (top: FMAC, bottom: TEX).

In [ ]:
source_box <- function(score, y_lab) {
    response_boxplot(all_scores, x = "cohort", y = score, y_lab = y_lab, facet = "dataset",
                     strip_size = 6, box_linewidth = 0.25, jitter_width = 0.2)
}

sfig6a <- source_box("FMAC_set1", "FMAC Score") + source_box("TEX_set1", "TEX Score") + plot_layout(nrow = 2)
sfig6a
save_plot(sfig6a, "SFig6A_fmac_tex_scores", width = 4.5, height = 3)

## Supplementary Figure 6B: PFS by LymphoMAP archetype

Archetype assigned to each cfRNA sample by the LymphoMAP classifier (Li et al. 2025; `archetype` in the sample
metadata). Patients infused before 2024-12-01.

In [ ]:
km_archetype <- km_by_group(filter(mdf, km_include), group = "archetype",
                            palette = LYMPHOMAP_COLORS, major_grid = FALSE)
km_archetype
save_plot(km_archetype, "SFig6B_pfs_by_lymphomap", width = 2, height = 2)